# 02 — Data Cleaning & Churn Definition
**Goal**: Make every cleaning decision explicit and documented. Define churn in a way that is behaviorally grounded, leakage-free, and defensible to evaluators.  
**Input**: `data/processed/01_merged_base.csv`  
**Output**: `data/processed/02_cleaned.csv` + `data/processed/02_churn_labels.csv` + `data/processed/data_cleaning_log.md`

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
PROC = Path('../data/processed')
FIG  = Path('../reports/figures')
os.makedirs(FIG,  exist_ok=True)
os.makedirs(PROC, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── Column name constants (adjust once here if your names differ) ───────────
KEY          = 'loyalty_number'
YEAR_COL     = 'year'
MONTH_COL    = 'month'
FLIGHTS_COL  = 'total_flights'
DISTANCE_COL = 'distance'
POINTS_ACC   = 'points_accumulated'
POINTS_RED   = 'points_redeemed'
CLV_COL      = 'clv'               # change to 'customer_lifetime_value' if needed
CARD_COL     = 'loyalty_card'      # tier column

# ── Cleaning log: every decision appended here ─────────────────────────────
cleaning_log = []
def log(msg):
    print(msg)
    cleaning_log.append(msg)

log('=== Data Cleaning Log ===')
print('Setup complete.')

---
## 1. Load Merged Base Table

In [ ]:
df = pd.read_csv(PROC / '01_merged_base.csv')

# Rebuild period column if lost during save
df['period'] = pd.to_datetime(
    df[YEAR_COL].astype(str) + '-' + df[MONTH_COL].astype(str).str.zfill(2)
)

TOTAL_ROWS    = len(df)
TOTAL_MEMBERS = df[KEY].nunique()

log(f'\n[LOAD] Rows: {TOTAL_ROWS:,} | Members: {TOTAL_MEMBERS:,}')
print(df.shape)
display(df.head(3))

---
## 2. Cleaning Decision 1 — Duplicate Rows
One row per member per year-month is the expected grain. Remove exact duplicates first.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=[KEY, YEAR_COL, MONTH_COL], keep='first')
dropped = before - len(df)
log(f'\n[DECISION 1] Duplicate rows (same member + year + month): {dropped:,} removed.')
log(f'  Rationale: Activity grain must be unique per member-month. '
    f'Keep first occurrence as conservative choice.')

---
## 3. Cleaning Decision 2 — Members With Activity Before Enrollment
These are data integrity violations and potential leakage sources.

In [ ]:
# Detect enrollment year column
enroll_col = None
for candidate in ['enrollment_year', 'enrolment_year', 'member_since_year']:
    if candidate in df.columns:
        enroll_col = candidate
        break

if enroll_col:
    # First flight year per member
    first_flight = (
        df[df[FLIGHTS_COL] > 0]
        .groupby(KEY)[YEAR_COL].min()
        .reset_index()
        .rename(columns={YEAR_COL: 'first_flight_year'})
    )
    enroll_map = df[[KEY, enroll_col]].drop_duplicates(KEY)
    check = enroll_map.merge(first_flight, on=KEY, how='inner')
    bad_members = check[check['first_flight_year'] < check[enroll_col]][KEY].tolist()

    before = df[KEY].nunique()
    df = df[~df[KEY].isin(bad_members)]
    after = df[KEY].nunique()

    log(f'\n[DECISION 2] Members active before enrollment date: {len(bad_members):,} removed.')
    log(f'  Members remaining: {after:,} (was {before:,})')
    log(f'  Rationale: Pre-enrollment activity is impossible — likely data entry errors '
        f'or system migration artefacts. Excluding prevents target leakage in the model.')
else:
    log('\n[DECISION 2] Enrollment year column not found — skipping pre-enrollment check.')
    log('  Action: manually verify column name in loyalty table.')

---
## 4. Cleaning Decision 3 — Negative & Impossible Values

In [ ]:
numeric_cols = [c for c in [FLIGHTS_COL, DISTANCE_COL, POINTS_ACC, POINTS_RED]
                if c in df.columns]

for col in numeric_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        df[col] = df[col].clip(lower=0)
        log(f'\n[DECISION 3] {col}: {neg_count:,} negative values clipped to 0.')
        log(f'  Rationale: Negative flight counts and distances are physically impossible. '
            f'Likely data entry errors. Clipping to 0 is conservative (vs dropping rows).')
    else:
        log(f'\n[DECISION 3] {col}: no negative values found — no action.')

In [ ]:
# ── CLV: handle zero / negative / null ──────────────────────────────────────
if CLV_COL in df.columns:
    clv_per_member = df.groupby(KEY)[CLV_COL].first()

    null_clv = clv_per_member.isnull().sum()
    zero_clv = (clv_per_member == 0).sum()
    neg_clv  = (clv_per_member < 0).sum()

    log(f'\n[DECISION 3b] CLV issues:')
    log(f'  Null CLV  : {null_clv:,}')
    log(f'  Zero CLV  : {zero_clv:,}')
    log(f'  Neg  CLV  : {neg_clv:,}')

    # Fill null CLV with median (not mean — CLV is right-skewed)
    clv_median = clv_per_member[clv_per_member > 0].median()
    df[CLV_COL] = df[CLV_COL].fillna(clv_median)
    df[CLV_COL] = df[CLV_COL].clip(lower=0)

    log(f'  Action: nulls filled with median ({clv_median:,.0f}), negatives clipped to 0.')
    log(f'  Rationale: Median imputation preserves distribution shape for a skewed metric.')

---
## 5. Cleaning Decision 4 — Outlier Treatment
Cap extreme values at 99.9th percentile. Do NOT drop — extreme flyers are real and analytically interesting.

In [ ]:
CAP_PERCENTILE = 0.999

caps = {}
for col in numeric_cols:
    cap_val = df[col].quantile(CAP_PERCENTILE)
    n_capped = (df[col] > cap_val).sum()
    df[col]  = df[col].clip(upper=cap_val)
    caps[col] = cap_val
    log(f'\n[DECISION 4] {col}: {n_capped:,} rows capped at 99.9th pct ({cap_val:,.1f}).')

log(f'  Rationale: Hard outliers distort distance-based features and model gradients. '
    f'Capping at 99.9th pct retains the signal (high-value flyers) while removing '
    f'likely data errors (e.g. single month with 10,000 flights).')

---
## 6. Churn Definition ⭐
This is the most important section. We define **three churn types** and justify why we use behavioural churn, not just formal cancellation.

### Definition chosen:
> A member is **churned at prediction month T** if they had ≥1 flight in the 12 months *before* T, but **zero flights AND zero point redemptions** in the 6 months *after* T.

**Why this definition:**
- Formal cancellation undercounts churn — many members go silent without cancelling
- 12-month lookback ensures we only label members who were genuinely active (not already dormant)
- 6-month forward window is operationally actionable — long enough to be meaningful, short enough to act on
- Including points redemption captures members who stopped flying but still engage via redemptions

In [ ]:
# ── Parameters — change these and re-run to test sensitivity ────────────────
LOOKBACK_MONTHS = 12   # must have been active in last N months to be at risk
FORWARD_MONTHS  = 6    # zero activity in next N months = churned

# Prediction cutoff: we predict churn AS OF this month
# Use end of 2016 so we have 2017–2018 data to validate forward window
PREDICTION_CUTOFF = pd.Timestamp('2016-12-01')

print(f'Prediction cutoff : {PREDICTION_CUTOFF.strftime("%Y-%m")}')
print(f'Lookback window   : {LOOKBACK_MONTHS} months before cutoff  '
      f'({(PREDICTION_CUTOFF - pd.DateOffset(months=LOOKBACK_MONTHS)).strftime("%Y-%m")} '
      f'to {PREDICTION_CUTOFF.strftime("%Y-%m")})')
print(f'Forward window    : {FORWARD_MONTHS} months after cutoff  '
      f'({(PREDICTION_CUTOFF + pd.DateOffset(months=1)).strftime("%Y-%m")} '
      f'to {(PREDICTION_CUTOFF + pd.DateOffset(months=FORWARD_MONTHS)).strftime("%Y-%m")})')

In [ ]:
# ── Step 1: Active members — had ≥1 flight in the lookback window ────────────
lookback_start = PREDICTION_CUTOFF - pd.DateOffset(months=LOOKBACK_MONTHS - 1)

lookback_data = df[
    (df['period'] >= lookback_start) &
    (df['period'] <= PREDICTION_CUTOFF)
]

active_members = (
    lookback_data.groupby(KEY)[FLIGHTS_COL]
    .sum()
    .reset_index()
    .query(f'{FLIGHTS_COL} >= 1')
    [KEY]
    .tolist()
)

print(f'Members active in lookback window ({lookback_start.strftime("%Y-%m")} '
      f'to {PREDICTION_CUTOFF.strftime("%Y-%m")}): {len(active_members):,}')

In [ ]:
# ── Step 2: Forward window — zero flights AND zero redemptions ───────────────
forward_start = PREDICTION_CUTOFF + pd.DateOffset(months=1)
forward_end   = PREDICTION_CUTOFF + pd.DateOffset(months=FORWARD_MONTHS)

forward_data = df[
    (df['period'] >= forward_start) &
    (df['period'] <= forward_end)   &
    (df[KEY].isin(active_members))
]

# Sum flights and redemptions per member in forward window
forward_agg_cols = {FLIGHTS_COL: 'sum'}
if POINTS_RED in df.columns:
    forward_agg_cols[POINTS_RED] = 'sum'

forward_activity = (
    forward_data
    .groupby(KEY)
    .agg(forward_agg_cols)
    .reset_index()
    .rename(columns={
        FLIGHTS_COL : 'fwd_flights',
        POINTS_RED  : 'fwd_redemptions'
    })
)

print(f'Active members with any forward data: {len(forward_activity):,}')

In [ ]:
# ── Step 3: Assign churn label ───────────────────────────────────────────────
churn_labels = pd.DataFrame({KEY: active_members})
churn_labels = churn_labels.merge(forward_activity, on=KEY, how='left')

# Fill NaN = member had NO rows at all in forward window = definitely churned
churn_labels['fwd_flights']       = churn_labels['fwd_flights'].fillna(0)
if 'fwd_redemptions' in churn_labels.columns:
    churn_labels['fwd_redemptions'] = churn_labels['fwd_redemptions'].fillna(0)

# Churn = zero flights AND (zero redemptions or redemptions col missing)
if 'fwd_redemptions' in churn_labels.columns:
    churn_labels['churned'] = (
        (churn_labels['fwd_flights']       == 0) &
        (churn_labels['fwd_redemptions']   == 0)
    ).astype(int)
else:
    churn_labels['churned'] = (churn_labels['fwd_flights'] == 0).astype(int)

churn_rate = churn_labels['churned'].mean()
n_churned  = churn_labels['churned'].sum()
n_retained = (churn_labels['churned'] == 0).sum()

print(f'\n=== CHURN LABEL SUMMARY ===')
print(f'Total eligible members : {len(churn_labels):,}')
print(f'Churned (label=1)      : {n_churned:,}  ({churn_rate:.1%})')
print(f'Retained (label=0)     : {n_retained:,}  ({1-churn_rate:.1%})')

log(f'\n[CHURN DEFINITION]')
log(f'  Active-in-lookback  : ≥1 flight in {LOOKBACK_MONTHS}m before {PREDICTION_CUTOFF.strftime("%Y-%m")}')
log(f'  Churned-in-forward  : 0 flights + 0 redemptions in {FORWARD_MONTHS}m after cutoff')
log(f'  Churn rate          : {churn_rate:.1%} ({n_churned:,} of {len(churn_labels):,} members)')

In [ ]:
# ── Sanity check: compare behavioural churn vs formal cancellation ───────────
cancel_cols = [c for c in df.columns if 'cancel' in c.lower()]

if cancel_cols:
    cancel_col = cancel_cols[0]
    member_cancel = df[[KEY, cancel_col]].drop_duplicates(KEY)
    churn_labels = churn_labels.merge(member_cancel, on=KEY, how='left')

    formally_cancelled   = (churn_labels[cancel_col].notna() &
                            (churn_labels[cancel_col] != 'Active')).sum()
    behav_not_cancelled  = ((churn_labels['churned'] == 1) &
                            (churn_labels[cancel_col] == 'Active')).sum()

    print(f'\n=== CHURN TYPE COMPARISON ===')
    print(f'Formally cancelled members          : {formally_cancelled:,}')
    print(f'Behaviourally churned but not cancelled: {behav_not_cancelled:,}')
    print(f'\n→ Using formal cancellation alone would MISS {behav_not_cancelled:,} at-risk members.')
    print(f'  This is the core argument for our richer churn definition.')

    log(f'\n[CHURN COMPARISON]')
    log(f'  Formally cancelled              : {formally_cancelled:,}')
    log(f'  Silent churners (missed by formal): {behav_not_cancelled:,}')
    log(f'  Behavioural churn captures {(behav_not_cancelled / max(n_churned, 1)):.0%} more at-risk members.')
else:
    print('No cancellation column found for comparison.')

In [ ]:
# ── Visualise churn label balance ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
labels_plot = ['Retained (0)', 'Churned (1)']
counts_plot = [n_retained, n_churned]
colors_plot = ['steelblue', 'crimson']
axes[0].bar(labels_plot, counts_plot, color=colors_plot, alpha=0.85, edgecolor='white')
axes[0].set_title('Churn label distribution')
axes[0].set_ylabel('Members')
for i, v in enumerate(counts_plot):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(churn_labels):.1%})',
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts_plot, labels=labels_plot, colors=colors_plot,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Churn rate')

plt.suptitle(f'Behavioural churn labels (lookback={LOOKBACK_MONTHS}m, forward={FORWARD_MONTHS}m)',
             fontsize=12)
plt.tight_layout()
plt.savefig(FIG / '02_churn_label_balance.png', bbox_inches='tight')
plt.show()

---
## 7. Churn Rate by Loyalty Card Tier
A key business insight — which tier has the most at-risk members?

In [ ]:
if CARD_COL in df.columns:
    member_card = df[[KEY, CARD_COL]].drop_duplicates(KEY)
    churn_by_tier = (
        churn_labels
        .merge(member_card, on=KEY, how='left')
        .groupby(CARD_COL)['churned']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'churn_rate', 'count': 'members'})
        .sort_values('churn_rate', ascending=False)
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(churn_by_tier[CARD_COL], churn_by_tier['churn_rate'],
                  color='steelblue', alpha=0.85, edgecolor='white')

    for bar, (_, row) in zip(bars, churn_by_tier.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f"{row['churn_rate']:.1%}\n(n={row['members']:,})",
                ha='center', fontsize=9)

    ax.set_ylabel('Churn rate')
    ax.set_xlabel('Loyalty card tier')
    ax.set_title('Churn rate by loyalty card tier')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    plt.tight_layout()
    plt.savefig(FIG / '02_churn_by_tier.png', bbox_inches='tight')
    plt.show()

    display(churn_by_tier)
else:
    print(f'Card tier column ({CARD_COL}) not found — adjust CARD_COL constant.')

---
## 8. Churn Rate by Province & Salary Band
Demographic breakdown — sets up segmentation analysis in notebook 06.

In [ ]:
# Adjust column names to match actual columns
SALARY_COL   = 'salary'
PROVINCE_COL = 'province'

demo_cols = [c for c in [SALARY_COL, PROVINCE_COL] if c in df.columns]

for col in demo_cols:
    member_demo = df[[KEY, col]].drop_duplicates(KEY)
    churn_demo = (
        churn_labels
        .merge(member_demo, on=KEY, how='left')
        .groupby(col)['churned']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'churn_rate', 'count': 'n'})
        .sort_values('churn_rate', ascending=False)
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(churn_demo[col].astype(str), churn_demo['churn_rate'],
           color='seagreen', alpha=0.8, edgecolor='white')
    ax.set_ylabel('Churn rate')
    ax.set_xlabel(col)
    ax.set_title(f'Churn rate by {col}')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIG / f'02_churn_by_{col}.png', bbox_inches='tight')
    plt.show()

    display(churn_demo)

---
## 9. Build the Cleaned Member-Level Table
Create a member-level snapshot as of the prediction cutoff. This is what the model will train on.

In [ ]:
# ── Only rows up to the prediction cutoff ────────────────────────────────────
df_train = df[df['period'] <= PREDICTION_CUTOFF].copy()

# ── Member-level aggregates (raw, before behavioral features) ────────────────
agg_dict = {}
for col in [FLIGHTS_COL, DISTANCE_COL, POINTS_ACC, POINTS_RED]:
    if col in df_train.columns:
        agg_dict[col] = 'sum'

member_summary = (
    df_train
    .groupby(KEY)
    .agg(agg_dict)
    .reset_index()
    .rename(columns={
        FLIGHTS_COL  : 'total_flights_historical',
        DISTANCE_COL : 'total_distance_historical',
        POINTS_ACC   : 'total_points_accumulated',
        POINTS_RED   : 'total_points_redeemed'
    })
)

# ── Months of membership ──────────────────────────────────────────────────────
tenure = (
    df_train
    .groupby(KEY)['period']
    .agg(first_month='min', last_month='max')
    .reset_index()
)
tenure['tenure_months'] = (
    (tenure['last_month'].dt.year  - tenure['first_month'].dt.year)  * 12 +
    (tenure['last_month'].dt.month - tenure['first_month'].dt.month) + 1
)
member_summary = member_summary.merge(tenure[[KEY, 'tenure_months']], on=KEY, how='left')

# ── Bring in static demographic columns from loyalty table ───────────────────
demo_static_cols = [KEY, CLV_COL, CARD_COL]
extra_demo = [c for c in ['salary', 'province', 'marital_status', 'education',
                           'gender', 'enrollment_year']
              if c in df_train.columns]
demo_static_cols += extra_demo
demo_static_cols = [c for c in demo_static_cols if c in df_train.columns]

demographics = df_train[demo_static_cols].drop_duplicates(KEY)
member_summary = member_summary.merge(demographics, on=KEY, how='left')

print(f'Member-level snapshot shape: {member_summary.shape}')
display(member_summary.head(3))

In [ ]:
# ── Attach churn labels ───────────────────────────────────────────────────────
# Only keep members in our active cohort (those with a churn label)
labeled_members = churn_labels[[KEY, 'churned']].copy()

cleaned = member_summary[member_summary[KEY].isin(labeled_members[KEY])].copy()
cleaned = cleaned.merge(labeled_members, on=KEY, how='inner')

print(f'Final cleaned table: {cleaned.shape}')
print(f'Churn rate in final table: {cleaned["churned"].mean():.1%}')
display(cleaned.head(3))

---
## 10. Final Null Check Before Saving

In [ ]:
null_summary = cleaned.isnull().sum()
null_summary = null_summary[null_summary > 0]

if len(null_summary) > 0:
    print('Remaining nulls:')
    display(null_summary.to_frame('null_count'))

    # For categorical columns: fill with mode
    cat_nulls = cleaned.select_dtypes(include='object').columns[
        cleaned.select_dtypes(include='object').isnull().any()
    ]
    for col in cat_nulls:
        mode_val = cleaned[col].mode()[0]
        cleaned[col] = cleaned[col].fillna(mode_val)
        log(f'\n[DECISION 5] {col}: nulls filled with mode ({mode_val}).')

    # For numeric columns: fill with median
    num_nulls = cleaned.select_dtypes(include='number').columns[
        cleaned.select_dtypes(include='number').isnull().any()
    ]
    for col in num_nulls:
        if col == 'churned':
            continue   # never impute the target
        med_val = cleaned[col].median()
        cleaned[col] = cleaned[col].fillna(med_val)
        log(f'\n[DECISION 5] {col}: nulls filled with median ({med_val:.2f}).')
else:
    print('No remaining nulls — table is clean.')
    log('\n[DECISION 5] No remaining nulls after all prior steps.')

print(f'\nFinal shape: {cleaned.shape}')
print(f'Final null count: {cleaned.isnull().sum().sum()}')

---
## 11. Save Outputs

In [ ]:
# ── Save cleaned member-level table ──────────────────────────────────────────
cleaned.to_csv(PROC / '02_cleaned.csv', index=False)
print(f'Saved: 02_cleaned.csv  ({cleaned.shape[0]:,} rows × {cleaned.shape[1]} cols)')

# ── Save churn labels separately (useful for model notebook) ─────────────────
churn_labels[[KEY, 'churned']].to_csv(PROC / '02_churn_labels.csv', index=False)
print(f'Saved: 02_churn_labels.csv  ({len(churn_labels):,} members)')

# ── Save full activity table (cutoff-filtered) for feature engineering ────────
df_train.to_csv(PROC / '02_activity_clean.csv', index=False)
print(f'Saved: 02_activity_clean.csv  ({len(df_train):,} rows)')

In [ ]:
# ── Write cleaning log to markdown file ──────────────────────────────────────
log_path = PROC / 'data_cleaning_log.md'
log_path.write_text('\n'.join(cleaning_log))
print(f'\nCleaning log saved to: {log_path}')
print('\nDecisions documented:')
for line in cleaning_log:
    if line.startswith('['):
        print(' ', line)

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  NOTEBOOK 02 COMPLETE')
print('='*55)
print(f'  Input rows (merged base)  : {TOTAL_ROWS:,}')
print(f'  Input members             : {TOTAL_MEMBERS:,}')
print(f'  Output members (labelled) : {len(cleaned):,}')
print(f'  Churn rate                : {cleaned["churned"].mean():.1%}')
print(f'  Features available        : {cleaned.shape[1] - 2}')
print(f'  Outputs saved to          : data/processed/')
print('='*55)
print('\nNext step → 03_eda_behavior_analysis.ipynb')